# relay_report

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): the analysis. `gateC1b` (FR-D2): part (i), effective, `bridgeFix` − `bridge1` in honest pooled share, a one-sided exact sign-flip over 8 paired boards; part (ii), safe, the 95th percentile of 2,000 paired bootstrap means of `bridgeFix` − `bridge1` in contamination below 0.05; and what the verdict decides for A1 (P23-7). `c1bSummaries` (FR-D3) adds part (i)'s robustness on raw honest welfare, H23-3 to H23-8 and, per arm, honest pooled share and contamination side by side with the bridge, relay and core measures; it refuses rows unless every planned cell is present. The report, the figure and synthetic rows are in `relay_write`.

In [ ]:
#| default_exp relay_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.bridge_mech import coreMeasures
from explRL.bridge_report import BRIDGE_COLS, _cell, _d, _lin, _ratio, _test
from explRL.colony4_report import CELL_KEYS
from explRL.colony_report import NAN, bootSeeds, cellVals
from explRL.relay_grid import ARMS23, N20
from explRL.relay_mech import RELAY_COLS
from explRL.relay_plan import C1B_PLAN
from explRL.rl19_measures import index

IND, CLEAN = 'independent', 'clean'


def planned23(plan=C1B_PLAN):
    "{(level, seed)} of every planned cell at N 64: the new cells with liars and clean, and C1's reused cells (36)."
    c = plan['cells']
    new = {(f'{a}|{f}', s) for f, cs in ((IND, c['liars']), (CLEAN, c['clean'])) for a, ss in cs.items() for s in ss}
    return new | {(f'{a}|{f}', s) for f, cs in ((IND, c['reused']['liars']), (CLEAN, c['reused']['clean'])) for a, ss in cs.items() for s in ss}


def upperBound(dp, nboot=2000, q=95, seed=0):
    "The q-th percentile of nboot paired bootstrap means of {(seed, N): difference} (pairs resampled whole); NaN when empty."
    v = np.asarray([x for x in dp.values() if np.isfinite(x)], float)
    if not len(v):
        return NAN
    rng = np.random.default_rng(seed)
    return float(np.percentile(v[rng.integers(0, len(v), (nboot, len(v)))].mean(1), q))


def gateC1b(rows, plan=C1B_PLAN, nboot=2000, metric='honestPooled'):
    """Gate C1b over the 8 paired boards (seeds 1800-1807). Part (i): bridgeFix - bridge1 in `metric`, one-sided exact sign-flip,
    passes at p < alpha with a positive estimate. Part (ii): bridgeFix - bridge1 in contamination, holds when upperBound
    (95th percentile of the paired bootstrap means) < margin. Verdict (P23-7): passes, noBridges (ii fails) or ask (i fails,
    ii holds), with what it decides."""
    G, s8 = plan['gate'], set(plan['seeds']['eight'])
    i = _test(_d(rows, 'bridgeFix', 'bridge1', metric, seeds=s8), nboot)
    i['passes'] = bool(i['p'] < G['alpha'] and i['estimate'] > 0)
    c = _d(rows, 'bridgeFix', 'bridge1', G['safety'], seeds=s8)
    ii = dict(_test(c, nboot, 'less'), upper=upperBound(c, plan['nboot'], G['quantile']), margin=G['margin'])
    ii['holds'] = bool(ii['upper'] < G['margin'])
    v = 'passes' if i['passes'] and ii['holds'] else 'noBridges' if not ii['holds'] else 'ask'
    return dict(metric=metric, i=i, ii=ii, passes=v == 'passes', verdict=v, decides=G['decides'][v])


def c1bSummaries(rows, plan=C1B_PLAN, nboot=None):
    """Every C1B_PLAN item from the full grid's rows (new and reused; FR-D2, FR-D3): Gate C1b with part (i)'s robustness on
    welfareHonest; on seeds 1800-1804, H23-3 (win-back, supported only with a positive coupling cost; truthConfirmable beside),
    H23-4 (filter gain over the bridgeFix boards), H23-5 (the 2 x 2), H23-6 (against the window), H23-7 (clean boards against
    squads) and H23-8 (coupled contamination); per arm, honest pooled share and contamination with bootstrap intervals, the
    columns and the core measures. ValueError unless every planned cell is present; other rows are left out."""
    want, nboot = planned23(plan), nboot or plan['nboot']
    key = lambda r: (r['level'], int(float(r['seed'])))
    rows = [r for r in rows if int(float(r['N'])) == N20 and key(r) in want]
    if miss := sorted(want - {key(r) for r in rows}):
        raise ValueError(f'{len(miss)} planned C1b cells missing, e.g. {miss[:4]}')
    gate, rob = gateC1b(rows, plan, nboot), gateC1b(rows, plan, nboot, 'welfareHonest')
    gate['robust'] = bool(np.sign(rob['i']['estimate']) == np.sign(gate['i']['estimate']))
    gate['welfareHonest'] = rob['i']
    s5 = set(plan['seeds']['five'])
    d = lambda a, b, m='honestPooled', f=IND: _d(rows, a, b, m, f, s5)
    fin = lambda a, m, f=IND: {k: v for k, v in cellVals(rows, a, f, m, [N20]).items() if np.isfinite(v)}
    fc, sc = d('bridgeFix', 'coupled'), d('squads', 'coupled')
    wb, wbCi = _ratio(fc, sc, nboot)
    H3 = dict(_test(_lin(fc, sc, 0.5), nboot), winBack=wb, winBackCi=wbCi, couplingCost=_test(sc, nboot), bridgeGain=_test(fc, nboot),
              truthConfirmable=bootSeeds({k: v for k, v in fin('bridgeFix', 'truthConfirmable').items() if k[0] in s5}, nboot=nboot))
    H3['supported'] = bool(H3['couplingCost']['estimate'] > 0 and wb > 0.5)
    H4 = _test(fin('bridgeFix', 'filterGain'), nboot)
    H4['supported'] = bool(H4['estimate'] > 0)
    e1, e2, r1, r2 = d('relayEvid', 'bridge1'), d('bridgeFix', 'bridgeRole'), d('bridgeRole', 'bridge1'), d('bridgeFix', 'relayEvid')
    avg = lambda a, b: {k: (a[k] + b[k]) / 2 for k in a if k in b}
    H5 = dict(evidence=_test(avg(e1, e2), nboot), role=_test(avg(r1, r2), nboot), interaction=_test(_lin(e2, e1, 1.0), nboot),
              cells={a: bootSeeds({k: v for k, v in fin(a, 'honestPooled').items() if k[0] in s5}, nboot=nboot)
                     for a in ('bridge1', 'relayEvid', 'bridgeRole', 'bridgeFix')})
    H6 = {f'{a} - mirrorWindow': _test(d(a, 'mirrorWindow'), nboot) for a in ('bridgeFix', 'relayEvid')}
    clean = sorted({a for a in plan['cells']['clean']} | set(plan['cells']['reused']['clean']), key=list(ARMS23).index)
    H7 = dict(perArm={a: {m: _cell(rows, a, CLEAN, m) for m in ('honestPooled', 'welfareHonest')} for a in clean},
              contrasts={f'{a} - squads': {m: _test(d(a, 'squads', m, CLEAN), nboot, 'less') for m in ('honestPooled', 'welfareHonest')}
                         for a in clean if a != 'squads'})
    coupled = [a for a in ARMS23 if ARMS23[a].get('couple')]
    H8 = dict(perArm={a: bootSeeds({k: v for k, v in fin(a, 'coupledContamination').items() if k[0] in s5}, nboot=nboot) for a in coupled},
              againstCoupled={a: _test(d(a, 'coupled', 'coupledContamination'), nboot, 'less') for a in coupled if a != 'coupled'})
    ix, arms = index(rows), {}
    for a in ARMS23:
        arms[a] = dict(honestPooled=bootSeeds(fin(a, 'honestPooled'), nboot=nboot), contamination=bootSeeds(fin(a, 'contamination'), nboot=nboot),
                       boards=len(fin(a, 'honestPooled')), columns={k: _cell(rows, a, IND, k) for k in CELL_KEYS + BRIDGE_COLS + RELAY_COLS},
                       core=coreMeasures(ix, a, IND), coreClean=coreMeasures(ix, a, CLEAN) if a in clean else None)
    return dict(gate=gate, hyp=dict(H23_3=H3, H23_4=H4, H23_5=H5, H23_6=H6, H23_7=H7, H23_8=H8), arms=arms, boards=len(rows))

In [ ]:
# test_gate_c1b (AC-9, FR-D2, FR-D3): on synthetic rows with known effects the gate returns the planted verdicts: a pass; a
# safety failure beside an effectiveness pass (noBridges: A1 trains without bridges); an effectiveness failure with safety
# holding (ask); the secondaries return their planted signs; a missing planned cell is refused
from explRL.relay_write import syntheticRows23
S = c1bSummaries(syntheticRows23(), nboot=200)
G = S['gate']
assert G['verdict'] == 'passes' and G['passes'] and G['i']['p'] == 1 / 256 and G['ii']['holds'] and G['robust']
assert abs(G['i']['estimate'] - 0.05) < 1e-9 and abs(G['ii']['upper'] - 0.0) < 1e-9 and 'dedicated bridge' in G['decides']
H = S['hyp']
assert H['H23_3']['supported'] and abs(H['H23_3']['winBack'] - 0.06 / 0.09) < 1e-9 and H['H23_4']['supported']
assert abs(H['H23_5']['evidence']['estimate'] - 0.025) < 1e-9 and abs(H['H23_5']['role']['estimate'] - 0.025) < 1e-9
assert abs(H['H23_5']['interaction']['estimate']) < 1e-9 and H['H23_6']['bridgeFix - mirrorWindow']['estimate'] < 0
assert set(H['H23_7']['contrasts']) == {'coupled - squads', 'bridge1 - squads', 'bridgeFix - squads', 'mirrorWindow - squads'}
assert H['H23_8']['againstCoupled']['bridge1']['estimate'] > 0 and S['boards'] == 69
assert abs(S['arms']['bridgeFix']['contamination'][0] - S['arms']['bridge1']['contamination'][0]) < 1e-9
unsafe = gateC1b(syntheticRows23(safety=0.08), nboot=200)
assert unsafe['verdict'] == 'noBridges' and unsafe['i']['passes'] and not unsafe['ii']['holds'] and 'without bridges' in unsafe['decides']
near = gateC1b(syntheticRows23(safety=0.04, safetyNoise=0.03), nboot=200)          # a noisy, nearly-unsafe design
assert near['ii']['upper'] > near['ii']['estimate'] and near['ii']['holds'] == (near['ii']['upper'] < 0.05)
weak = gateC1b(syntheticRows23(ev=0.0, role=0.0), nboot=200)
assert weak['verdict'] == 'ask' and not weak['i']['passes'] and weak['ii']['holds']
try:
    c1bSummaries(syntheticRows23()[1:]); raise AssertionError('a missing cell must be refused')
except ValueError:
    pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()